# Phase 3 — Clustering & Labeling

In [6]:

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:

!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.1 sentence-transformers scikit-learn pyarrow openai -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"



In [8]:

CATEGORY_NAME         = "Electronics_part1"
INPUT_PARQUET         = "/content/drive/MyDrive/outputs_electronics_cleaning/cleaned_part1_step2.parquet"
OUTPUT_DIR            = "/content/drive/MyDrive/outputs_electronics_cleaning"
OUTPUT_PARQUET        = f"{OUTPUT_DIR}/labeled_{CATEGORY_NAME}.parquet"
REPORT_PATH           = f"{OUTPUT_DIR}/reports/hard_labeling_report_{CATEGORY_NAME}.txt"

# Local
LOCAL_EMBED           = "/content/embeddings_1m.npy"
LOCAL_SAMPLE_META     = "/content/sample_meta.parquet"

TO_LABEL_CSV          = f"{OUTPUT_DIR}/to_label_800.csv"
LABELED_CSV           = f"{OUTPUT_DIR}/labeled_800.csv"


SAMPLE_SIZE           = 1_000_000
N_CLUSTERS            = 100
SENTENCES_PER_CLUSTER = 8
EMBED_BATCH_SIZE      = 1024
RANDOM_SEED           = 42

os.makedirs(f"{OUTPUT_DIR}/reports", exist_ok=True)


In [9]:
# Sample 1M câu → pandas
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("Phase3_Labeling")
    .master("local[*]")
    .config("spark.driver.memory",                  "40g")
    .config("spark.driver.maxResultSize",           "10g")
    .config("spark.sql.shuffle.partitions",         "400")
    .config("spark.sql.adaptive.enabled",           "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.parquet.compression.codec",  "gzip")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

df_full = spark.read.parquet(INPUT_PARQUET)
total_count = df_full.count()
print(f"Tổng câu Phase-2: {total_count:,}")

frac = min(1.0, SAMPLE_SIZE / total_count)
df_sample_spark = df_full.sample(fraction=frac, seed=RANDOM_SEED)
actual_sample   = df_sample_spark.count()
print(f"Đã sample: {actual_sample:,} (fraction={frac:.4f})")

df_meta = (
    df_sample_spark
    .select("parent_asin", "review_id", "sentence_id", "sentence_text", "rating")
    .toPandas()
    .reset_index(drop=True)
)

df_meta.to_parquet(LOCAL_SAMPLE_META, index=False)
print(f"Meta lưu local: {LOCAL_SAMPLE_META} ({len(df_meta):,} hàng)")
df_meta.head(3)

Tổng câu Phase-2: 39,063,570
Đã sample: 999,543 (fraction=0.0256)
Meta lưu local: /content/sample_meta.parquet (999,543 hàng)


,parent_asin,review_id,sentence_id,sentence_text,rating
0,B0163JVYEW,1503,14,i wash dishes and [GENERIC_NOUN] of that natur...,5.0
1,B00B9JR5H2,13346,1,the setup on this player was so quick i was li...,5.0
2,B00B9JR5H2,13346,5,needed it for mainly to stream netflix but thi...,5.0


In [10]:
#  Embed = all-MiniLM-L6-v2
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# df_meta = pd.read_parquet(LOCAL_SAMPLE_META)

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device,
)


texts_clean = [
    t.replace("[GENERIC_NOUN]", "thing").replace("[DOMAIN_NOISE]", "item")
    for t in df_meta["sentence_text"].tolist()
]

print(f"Bắt đầu encode {len(texts_clean):,} câu...")
embeddings = model.encode(
    texts_clean,
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    precision="float32",
)
print(f"Embeddings shape: {embeddings.shape}")

del model
torch.cuda.empty_cache()

np.save(LOCAL_EMBED, embeddings)
print(f"Saved: {LOCAL_EMBED} ({embeddings.nbytes / 1e9:.2f} GB)")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Bắt đầu encode 999,543 câu...


Batches:   0%|          | 0/977 [00:00<?, ?it/s]

Embeddings shape: (999543, 384)
Saved: /content/embeddings_1m.npy (1.54 GB)


In [11]:
#  MiniBatchKMeans 100 cụm → chọn 8 câu/cụm
from sklearn.cluster import MiniBatchKMeans

# embeddings = np.load(LOCAL_EMBED)
# df_meta    = pd.read_parquet(LOCAL_SAMPLE_META)

print(f"Clustering {len(embeddings):,} vectors → {N_CLUSTERS} clusters...")
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_SEED,
    n_init=10,
    batch_size=20_000,
    max_iter=500,
    verbose=0,
)
cluster_labels = kmeans.fit_predict(embeddings)
df_meta["cluster"] = cluster_labels

cs = df_meta["cluster"].value_counts()
print(f"Cluster sizes — min: {cs.min()}, max: {cs.max()}, mean: {cs.mean():.0f}")

sampled_rows = []
for cid in range(N_CLUSTERS):
    cdf = df_meta[df_meta["cluster"] == cid]
    n   = min(SENTENCES_PER_CLUSTER, len(cdf))
    sampled_rows.append(cdf.sample(n=n, random_state=cid))

df_to_label = pd.concat(sampled_rows).reset_index(drop=True)
df_to_label["triplets"] = None

df_to_label.to_csv(TO_LABEL_CSV, index=False)
print(f"\nTổng câu cần gán nhãn: {len(df_to_label)}")
print(f"To-label CSV: {TO_LABEL_CSV}")
df_to_label[["cluster", "sentence_text"]].head(8)

Clustering 999,543 vectors → 100 clusters...
Cluster sizes — min: 4415, max: 19388, mean: 9995

Tổng câu cần gán nhãn: 800
To-label CSV: /content/drive/MyDrive/outputs_electronics_cleaning/to_label_800.csv


,cluster,sentence_text
0,0,i assumed alphabetical but i put in 3 files an...
1,0,have no fear the roadmate will direct you righ...
2,0,even the label is wrong
3,0,i can t take it anymore
4,0,make your decision wisely
5,0,2 but all of the mixed [GENERIC_NOUN] scared m...
6,0,the only [GENERIC_NOUN] have been through movi...
7,0,goodbye reolink


In [12]:

import ast

df_labeled_final = pd.read_csv(LABELED_CSV)

def safe_parse(x):
    if pd.isna(x) or str(x).strip() in ("", "nan"):
        return []
    s = str(x).strip()

    try:
        val = json.loads(s)
        return val if isinstance(val, list) else []
    except Exception:
        pass

    try:
        val = ast.literal_eval(s)
        return val if isinstance(val, list) else []
    except Exception:
        return []

def normalize_triplet(t):
    """Chuẩn hóa triplet về dạng [aspect, opinion, sentiment_int]."""
    if isinstance(t, list) and len(t) == 3:
        try:
            sentiment = int(t[2])
            if sentiment in (0, 1, 2):
                return [str(t[0]), str(t[1]), sentiment]
        except (ValueError, TypeError):
            pass
    elif isinstance(t, dict):
        aspect   = t.get("aspect_phrase") or t.get("aspect") or t.get("a")
        opinion  = t.get("opinion_phrase") or t.get("opinion") or t.get("o")
        sentiment = t.get("sentiment_int") or t.get("sentiment") or t.get("s")
        if aspect is not None and opinion is not None and sentiment is not None:
            try:
                sentiment = int(sentiment)
                if sentiment in (0, 1, 2):
                    return [str(aspect), str(opinion), sentiment]
            except (ValueError, TypeError):
                pass
    return None

def validate_triplets(triplets):
    """Chuẩn hóa và giữ lại triplet hợp lệ dạng [aspect, opinion, sentiment]."""
    result = []
    for t in triplets:
        normalized = normalize_triplet(t)
        if normalized is not None:
            result.append(normalized)
    return result

df_labeled_final["triplets"] = (
    df_labeled_final["triplets"]
    .apply(safe_parse)
    .apply(validate_triplets)
)

valid   = df_labeled_final["triplets"].apply(lambda x: len(x) > 0).sum()
invalid = len(df_labeled_final) - valid
print(f"Validation OK — có nhãn: {valid}, không có nhãn: {invalid}")
df_labeled_final[["sentence_text", "triplets"]].head(10)


Validation OK — có nhãn: 381, không có nhãn: 419


,sentence_text,triplets
0,i assumed alphabetical but i put in 3 files an...,[]
1,have no fear the roadmate will direct you righ...,[]
2,even the label is wrong,"[[label, wrong, 0]]"
3,i can t take it anymore,[]
4,make your decision wisely,[]
5,2 but all of the mixed [GENERIC_NOUN] scared m...,[]
6,the only [GENERIC_NOUN] have been through movi...,[]
7,goodbye reolink,[]
8,the [GENERIC_NOUN] is the mobile app,[]
9,the ushare app paired with it without [GENERIC...,"[[sim card, easy to use , 2]]"


In [14]:
import json
from pyspark.sql.types import StringType

label_map = {
    (int(row["review_id"]), int(row["sentence_id"])): json.dumps(row["triplets"])
    for _, row in df_labeled_final.iterrows()
}
bc_label_map = spark.sparkContext.broadcast(label_map)
print(f"Labeled entries in broadcast map: {len(label_map)}")

@F.udf(StringType())
def get_triplets_udf(review_id, sentence_id):
    key = (int(review_id), int(sentence_id))
    return bc_label_map.value.get(key, "[]")


# df_full = spark.read.parquet(INPUT_PARQUET)

df_out = (
    df_full
    .withColumn("triplets",      get_triplets_udf(F.col("review_id"), F.col("sentence_id")))
    .withColumn("category_name", F.lit(CATEGORY_NAME))

    .select("parent_asin", "sentence_id", "sentence_text", "rating", "triplets", "category_name")
)

print("Đang ghi output parquet...")
df_out.repartition(200).write.mode("overwrite").parquet(OUTPUT_PARQUET)
print(f"Saved: {OUTPUT_PARQUET}")

Labeled entries in broadcast map: 800
Đang ghi output parquet...
Saved: /content/drive/MyDrive/outputs_electronics_cleaning/labeled_Electronics_part1.parquet


In [19]:

from pyspark.sql.types import ArrayType
from pyspark.sql.functions import from_json, col, size, explode_outer, sum as F_sum

triplet_schema = ArrayType(ArrayType(StringType()))

df_stats = (
    spark.read.parquet(OUTPUT_PARQUET)
    .withColumn("triplet_list", from_json(col("triplets"), triplet_schema))
    .cache()
)

total_sentences   = df_stats.count()
labeled_sentences = df_stats.filter(size("triplet_list") > 0).count()
empty_sentences   = 800 - labeled_sentences
multi_triplet     = df_stats.filter(size("triplet_list") > 1).count()
total_triplets_v  = df_stats.select(F_sum(size("triplet_list"))).collect()[0][0] or 0


df_exploded = (
    df_stats
    .filter(size("triplet_list") > 0)
    .select(explode_outer("triplet_list").alias("triplet"))
    .select(col("triplet")[2].cast("string").alias("sentiment"))
)
sent_map  = {r["sentiment"]: r["count"] for r in df_exploded.groupBy("sentiment").count().collect()}
neg_count = sent_map.get("0", 0)
neu_count = sent_map.get("1", 0)
pos_count = sent_map.get("2", 0)


def get_dir_size(path):
    total = 0
    for dp, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(dp, f))
    return total

def fmt_size(b):
    if b >= 1 << 30: return f"{b/(1<<30):.2f} GB"
    if b >= 1 << 20: return f"{b/(1<<20):.2f} MB"
    return f"{b/(1<<10):.2f} KB"

out_size_str = fmt_size(get_dir_size(OUTPUT_PARQUET))

report = f"""Labeling Report: {CATEGORY_NAME}

[Dataset]

  Số câu được gán nhãn (triplets ≠ [])  : {labeled_sentences:>12,}
  Số câu không có triplet ([])           : {empty_sentences:>12,}
  Số câu có nhiều hơn 1 triplet          : {multi_triplet:>12,}
  Tổng số triplet                        : {total_triplets_v:>12,}

[Phân phối Sentiment]
  Negative (0) : {neg_count:>10,}
  Neutral  (1) : {neu_count:>10,}
  Positive (2) : {pos_count:>10,}

[Output]
  Parquet  : {OUTPUT_PARQUET}
  Size     : {out_size_str}
  Report   : {REPORT_PATH}
"""

print(report)
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report)
print(f"Report saved: {REPORT_PATH}")

df_stats.unpersist()

Labeling Report: Electronics_part1 

[Dataset]
  
  Số câu được gán nhãn (triplets ≠ [])  :          381
  Số câu không có triplet ([])           :          419
  Số câu có nhiều hơn 1 triplet          :           87
  Tổng số triplet                        :          506

[Phân phối Sentiment]
  Negative (0) :        172
  Neutral  (1) :         26
  Positive (2) :        308

[Output]
  Parquet  : /content/drive/MyDrive/outputs_electronics_cleaning/labeled_Electronics_part1.parquet
  Size     : 1.48 GB
  Report   : /content/drive/MyDrive/outputs_electronics_cleaning/reports/hard_labeling_report_Electronics_part1.txt

Report saved: /content/drive/MyDrive/outputs_electronics_cleaning/reports/hard_labeling_report_Electronics_part1.txt


DataFrame[parent_asin: string, sentence_id: int, sentence_text: string, rating: double, triplets: string, category_name: string, triplet_list: array<array<string>>]

In [20]:

from pyspark.sql.types import ArrayType
from pyspark.sql.functions import from_json, col, size

triplet_schema2 = ArrayType(ArrayType(StringType()))
df_preview = spark.read.parquet(OUTPUT_PARQUET)

print("Schema:")
df_preview.printSchema()

print("\n── Câu có nhãn (sample 10) ──")
(
    df_preview
    .withColumn("tl", from_json(col("triplets"), triplet_schema2))
    .filter(size("tl") > 0)
    .select("parent_asin", "sentence_text", "rating", "triplets", "category_name")
    .show(10, truncate=90)
)

print("\n── Câu không có nhãn (sample 5) ──")
(
    df_preview
    .filter(col("triplets") == "[]")
    .select("sentence_text", "rating")
    .show(5, truncate=90)
)

spark.stop()
print("Done — Spark stopped.")

Schema:
root
 |-- parent_asin: string (nullable = true)
 |-- sentence_id: integer (nullable = true)
 |-- sentence_text: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- triplets: string (nullable = true)
 |-- category_name: string (nullable = true)


── Câu có nhãn (sample 10) ──
+-----------+------------------------------------------------------------------------------------------+------+------------------------------------------------------------------------------------------+-----------------+
|parent_asin|                                                                             sentence_text|rating|                                                                                  triplets|    category_name|
+-----------+------------------------------------------------------------------------------------------+------+------------------------------------------------------------------------------------------+-----------------+
| B077QDKRC7|the savings alone over t